# Extrator de agendamentos em português: LoRA no Qwen2.5-1.5B vs Gemini

Um modelo pequeno lê a mensagem de WhatsApp que o cliente mandou para um negócio e devolve o pedido de agendamento em JSON
(intenção, serviço, profissional, data, hora, período). Aqui o Qwen2.5-1.5B-Instruct é ajustado com LoRA e comparado com
ele mesmo sem ajuste e com o Gemini 2.5 Flash.

Código e dados: [github.com/HowLuks/extrator-agendamentos-ptbr](https://github.com/HowLuks/extrator-agendamentos-ptbr).

**Para rodar:** GPU T4, internet ligada e o secret `HF_TOKEN` (token do Hugging Face com permissão de escrita).

In [ ]:
!git clone -q https://github.com/HowLuks/extrator-agendamentos-ptbr.git
%cd extrator-agendamentos-ptbr
!pip install -q "trl>=0.20" "peft>=0.13"

In [ ]:
import json, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from extrator.dados import ler_jsonl
from extrator.prompt import mensagens_extracao, resposta_esperada, ler_resposta
from extrator.metricas import relatorio, tabela

MODELO_BASE = "Qwen/Qwen2.5-1.5B-Instruct"
HF_REPO = "HowLuks/qwen2.5-1.5b-extrator-agendamentos-ptbr"
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None
    print("Sem HF_TOKEN: o modelo não será publicado.")

## Dados
Treino: todas as mensagens já escritas. Teste: só as revisadas à mão (se ainda não houver nenhuma, usa todas, com aviso).

In [ ]:
treino = [ex for ex in ler_jsonl("dados/treino.jsonl") if ex["mensagem"]]
todos_teste = [ex for ex in ler_jsonl("dados/teste.jsonl") if ex["mensagem"] and not ex.get("descartado")]
teste = [ex for ex in todos_teste if ex.get("revisado")]
if not teste:
    print("AVISO: nenhum exemplo de teste revisado; avaliando no teste bruto.")
    teste = todos_teste
print(len(treino), "exemplos de treino,", len(teste), "de teste")

def para_sft(ex):
    return {"prompt": mensagens_extracao(ex), "completion": [{"role": "assistant", "content": resposta_esperada(ex["rotulo"])}]}

ds = Dataset.from_list([para_sft(ex) for ex in treino]).train_test_split(test_size=0.1, seed=0)

## Treino com LoRA
A perda é calculada só na resposta (o JSON), não no prompt com o catálogo.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
modelo = AutoModelForCausalLM.from_pretrained(MODELO_BASE, torch_dtype=torch.float16, device_map="auto")

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")
config = SFTConfig(
    output_dir="/kaggle/working/saida", num_train_epochs=3, per_device_train_batch_size=4,
    gradient_accumulation_steps=4, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
    logging_steps=10, eval_strategy="epoch", save_strategy="no", fp16=True, max_length=1024, report_to="none",
)
trainer = SFTTrainer(model=modelo, args=config, train_dataset=ds["train"], eval_dataset=ds["test"],
                     processing_class=tokenizer, peft_config=lora)
trainer.train()
modelo = trainer.model

## Avaliação
O mesmo prompt para todos os modelos. O modelo base é avaliado desligando o adaptador LoRA.

In [ ]:
tokenizer.padding_side = "left"

@torch.no_grad()
def prever(exemplos, lote=16):
    modelo.eval()
    saidas = []
    for i in range(0, len(exemplos), lote):
        textos = [tokenizer.apply_chat_template(mensagens_extracao(ex), tokenize=False, add_generation_prompt=True)
                  for ex in exemplos[i:i + lote]]
        entrada = tokenizer(textos, return_tensors="pt", padding=True).to(modelo.device)
        gerado = modelo.generate(**entrada, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        saidas += tokenizer.batch_decode(gerado[:, entrada["input_ids"].shape[1]:], skip_special_tokens=True)
    return saidas

resp_lora = prever(teste)
with modelo.disable_adapter():
    resp_base = prever(teste)

resultados = {
    "qwen_lora": relatorio(teste, [ler_resposta(r) for r in resp_lora]),
    "qwen_base": relatorio(teste, [ler_resposta(r) for r in resp_base]),
}
for nome, rel in resultados.items():
    print(f"\n### {nome}\n" + tabela(rel))

In [ ]:
# Gemini: métricas geradas localmente por scripts/avaliar_gemini.py (no mesmo teste revisado)
try:
    resultados["gemini"] = json.load(open("resultados/gemini_metricas.json"))
    print("### gemini\n" + tabela(resultados["gemini"]))
except FileNotFoundError:
    print("Sem resultados do Gemini no repositório.")

json.dump(resultados, open("/kaggle/working/metricas.json", "w"), indent=2, ensure_ascii=False)
with open("/kaggle/working/previsoes.jsonl", "w") as f:
    for ex, rl, rb in zip(teste, resp_lora, resp_base):
        f.write(json.dumps({"id": ex["id"], "lora": rl, "base": rb}, ensure_ascii=False) + "\n")

## Publicação no Hugging Face
Sobe só o adaptador LoRA (alguns MB) e um model card com a tabela de resultados.

In [ ]:
if HF_TOKEN:
    from huggingface_hub import HfApi
    modelo.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    linhas = ["| modelo | exata | intencao | servico | profissional | data | hora | periodo |", "|---|---|---|---|---|---|---|---|"]
    for nome, rel in resultados.items():
        g = rel["geral"]
        linhas.append(f"| {nome} | " + " | ".join(f"{g[c]:.1%}" for c in ["exata", "intencao", "servico", "profissional", "data", "hora", "periodo"]) + " |")
    card = f'''---
language: pt
license: apache-2.0
base_model: {MODELO_BASE}
library_name: peft
tags: [lora, information-extraction, portuguese, whatsapp]
---

# Extrator de agendamentos em português (LoRA sobre Qwen2.5-1.5B-Instruct)

Lê a mensagem de WhatsApp de um cliente e devolve o pedido de agendamento em JSON: intenção, serviço, profissional,
data, hora e período. O catálogo do negócio (serviços e profissionais) e a data de hoje vão no prompt de sistema.

Treinado com {len(treino)} mensagens sintéticas em português e avaliado em {len(teste)} mensagens de teste.
Dados, prompt e código: https://github.com/HowLuks/extrator-agendamentos-ptbr

## Resultados (teste)

{chr(10).join(linhas)}

"exata" é a fração de mensagens com todos os campos certos.

## Limitações

As mensagens foram escritas por LLMs (Gemini e Claude) a partir de rótulos sorteados; mensagens reais tendem a ser mais
bagunçadas. O modelo não guarda contexto de conversa: cada mensagem é lida sozinha.
'''
    HfApi().upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=HF_REPO, token=HF_TOKEN)
    print("Publicado em https://huggingface.co/" + HF_REPO)